# Example Core Scalar Surface

Canonical core-scalar example notebook for the `arb_core`, `acb_core`, `arf`, `acf`, `fmpr`, `fmpzi`, and `arb_fpwrap` public surfaces.

## Scope

This notebook is the canonical example surface for `example_core_scalar_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_core_scalar_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_core_scalar_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view.

In [ ]:
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)

## Object/Input Construction

Construct representative real intervals, complex boxes, floating arrays, integer-interval arrays, and fpwrap point inputs.

In [ ]:
import jax.numpy as jnp
from arbplusjax import acb_core, api, double_interval as di

real_interval = di.interval(jnp.array([0.2, 0.5, 1.0], dtype=jnp.float64), jnp.array([0.25, 0.6, 1.1], dtype=jnp.float64))
complex_box = acb_core.acb_box(
    di.interval(jnp.array([0.2, 0.4], dtype=jnp.float64), jnp.array([0.25, 0.5], dtype=jnp.float64)),
    di.interval(jnp.array([-0.2, 0.1], dtype=jnp.float64), jnp.array([-0.1, 0.2], dtype=jnp.float64)),
)
float_a = jnp.array([1.0, 2.0, 3.0], dtype=jnp.float32)
float_b = jnp.array([0.5, 1.5, 2.5], dtype=jnp.float32)
complex_a = jnp.array([1.0 + 0.5j, 2.0 - 0.25j], dtype=jnp.complex64)
int_interval_a = jnp.array([[1, 2], [3, 5], [8, 13]], dtype=jnp.int64)
int_interval_b = jnp.array([[2, 3], [5, 8], [13, 21]], dtype=jnp.int64)
fpwrap_real = jnp.array([0.1, 0.4, 0.9], dtype=jnp.float32)
display({'real_interval_shape': real_interval.shape, 'complex_box_shape': complex_box.shape})

## Direct Usage

Run the public API directly on representative core scalar families.

In [ ]:
results = {
    'arb_exp_basic': api.eval_interval('arb_exp', real_interval, mode='basic', dtype='float64'),
    'acb_sin_basic': api.eval_interval('acb_sin', complex_box, mode='basic', dtype='float64'),
    'arf_add': api.eval_point('arf_add', float_a, float_b),
    'acf_mul': api.eval_point('acf_mul', complex_a, complex_a),
    'fmpr_mul': api.eval_point('fmpr_mul', float_a, float_b),
    'fmpzi_add': api.eval_point('fmpzi_add', int_interval_a, int_interval_b),
    'arb_fpwrap_double_exp': api.eval_point('arb_fpwrap_double_exp', fpwrap_real),
}
display(results)

## Parameter/Value Sweeps

Sweep representative scalar families over values and modes using the existing harness profile.

In [ ]:
profile_dir = EXAMPLE_OUTPUT_ROOT / ('cpu_profile' if JAX_MODE == 'cpu' else 'gpu_profile')
run([
    PYTHON, 'benchmarks/run_harness_profile.py',
    '--name', f'example_core_scalar_{JAX_MODE}',
    '--outdir', str(profile_dir),
    '--functions', 'exp,log,sqrt,sin,cos,tan,sinh,cosh,tanh',
    '--samples', '64,128',
    '--seeds', '0,1',
    '--jax-mode', JAX_MODE,
    '--jax-dtype', JAX_DTYPE,
    '--prec-bits', '128',
])
profile_csv = profile_dir / 'profile_summary.csv'
profile_df = pd.read_csv(profile_csv)
display(profile_df.head(20))

## Validation Summary

Run the existing scalar chassis/API tests and summarize the result.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_core_scalar_api_contracts.py',
    'tests/test_arb_core_chassis.py',
    'tests/test_acb_core_chassis.py',
    'tests/test_arf_chassis.py',
    'tests/test_acf_chassis.py',
    'tests/test_fmpr_chassis.py',
    'tests/test_fmpzi_chassis.py',
    'tests/test_arb_fpwrap_chassis.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)

## Benchmark Summary

Run representative scalar benchmark entrypoints and summarize cold/warm/recompile behavior.

In [ ]:
bench_dir = EXAMPLE_OUTPUT_ROOT / 'benchmark_artifacts'
bench_dir.mkdir(parents=True, exist_ok=True)
run([PYTHON, 'benchmarks/benchmark_arf.py', '--samples', '4096', '--which', 'add', '--runs', '3', '--output', str(bench_dir / 'benchmark_arf.json')])
run([PYTHON, 'benchmarks/benchmark_arb_fpwrap.py', '--samples', '4096', '--which', 'exp', '--runs', '3', '--output', str(bench_dir / 'benchmark_arb_fpwrap.json')])
bench_payloads = []
for path in sorted(bench_dir.glob('*.json')):
    payload = json.loads(path.read_text())
    for row in payload['records']:
        bench_payloads.append(row)
bench_df = pd.DataFrame(bench_payloads)
display(bench_df[['operation', 'cold_time_s', 'warm_time_s', 'recompile_time_s']])

## Comparison Summary

Where reference software is available, use the existing compare scripts. If local C reference libraries are absent, note that explicitly.

In [ ]:
compare_cmds = [
    [PYTHON, 'benchmarks/compare_arb_core.py', '--samples', '256', '--output', str(EXAMPLE_OUTPUT_ROOT / 'compare_arb_core.json')],
    [PYTHON, 'benchmarks/compare_acb_core.py', '--samples', '256', '--output', str(EXAMPLE_OUTPUT_ROOT / 'compare_acb_core.json')],
]
for cmd in compare_cmds:
    try:
        completed = subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=True, check=True)
        print(completed.stdout)
    except subprocess.CalledProcessError as exc:
        print('comparison unavailable or failed for', cmd[1])
        print(exc.stdout)
        print(exc.stderr)

## Plots

Plot backend timing and containment summaries from the harness profile.

In [ ]:
plot_df = profile_df.copy()
for col in ['time_ms', 'containment_rate', 'mean_abs_err']:
    if col in plot_df.columns:
        plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
summary = plot_df.groupby('backend', dropna=False)[['time_ms', 'containment_rate']].mean(numeric_only=True).sort_values('time_ms')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary['time_ms'].plot(kind='bar', ax=axes[0], title='Mean Time (ms)', color='#b85c38')
summary['containment_rate'].plot(kind='bar', ax=axes[1], title='Mean Containment', color='#41535d')
fig.tight_layout()
plt.show()

## Optional Diagnostics

Use the matrix/compile diagnostics tools only when compile traces or memory deltas are needed. The scalar tranche here keeps those optional.